# 02 — Financial Health Dashboard

Three metrics across Tesla, BYD and Ford for 2022–2024, from
`data/fundamentals.pkl` (notebook 01 must be run first).

All figures are in **US dollars**: BYD's renminbi statements are translated in
notebook 01 before they reach here, so the free cash flow panel compares like
with like. It previously did not, which made BYD's bar roughly seven times too
tall.

In [1]:
import sys

sys.path.append("..")

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import config

fundamentals = pd.read_pickle("../data/fundamentals.pkl")
YEARS = config.ANALYSIS_YEARS
COLORS = config.COLORS

METRICS = [
    ("npm", "Net Profit Margin (%)", 1),
    ("free_cash_flow", "Free Cash Flow (USD Billions)", 1e9),
    ("roa", "Return on Assets (%)", 1),
]

fig = make_subplots(rows=1, cols=3, subplot_titles=[title for _, title, _ in METRICS])

for col, (field, title, scale) in enumerate(METRICS, start=1):
    for name in config.COMPANIES:
        series = fundamentals.query("company == @name").set_index("year")[field] / scale
        fig.add_trace(go.Scatter(
            x=series.index,
            y=series.values,
            name=name,
            mode="lines+markers",
            line=dict(color=COLORS[name], width=2),
            marker=dict(size=8),
            legendgroup=name,
            showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(
    title_text="Financial Health — Tesla vs BYD vs Ford (2022–2024, USD)",
    template="plotly_white",
    height=450,
    width=1200,
    hovermode="x unified",
)
for col in range(1, 4):
    fig.update_xaxes(tickmode="array", tickvals=YEARS,
                     ticktext=[str(y) for y in YEARS], row=1, col=col)

fig.show()

In [2]:
snapshot = (fundamentals.query("year == @config.LAST_YEAR")
            .set_index("company")[["npm", "roa", "free_cash_flow", "ccr"]]
            .copy())
snapshot["free_cash_flow"] /= 1e9
snapshot.columns = ["NPM (%)", "ROA (%)", "FCF ($B)", "CCR"]
print(f"=== {config.LAST_YEAR} SNAPSHOT (USD) ===")
print(snapshot.round(2).to_string())

=== 2024 SNAPSHOT (USD) ===
         NPM (%)  ROA (%)  FCF ($B)   CCR
company                                  
Tesla       7.26     5.81      3.58  2.10
BYD         5.18     5.22      5.02  3.32
Ford        3.18     2.06      6.74  2.62


## What the three panels show

- **Net Profit Margin** — Tesla's halves, from 15.4% to 7.3%, as the 2023 price
  war works through a full year. BYD's rises steadily from 3.9% to 5.2%; Ford's
  recovers from a loss to 3.2%. The three converge into a band between 3% and 7%,
  which is the single clearest picture of the EV transition compressing everyone
  towards the same place.
- **Free Cash Flow** — the panel the currency bug distorted most. Corrected,
  **Ford generates the most cash of the three** ($6.7B), BYD is second ($5.0B) and
  Tesla last ($3.6B). BYD's is roughly flat across the window rather than
  collapsing from a huge base.
- **Return on Assets** — Tesla's falls from 15.2% to 5.8%, a steeper drop than its
  margin, because the asset base grew 48% while earnings fell. BYD overtakes
  nothing yet but closes most of the gap, reaching 5.2% against Tesla's 5.8%.